In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
checkPointPath = str(Path.cwd()) + r"\out\modeling\checkpoints\classification_finetuning\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
len("EFFABCD01234567890123456789012345678901234567890")

48

In [3]:
from ble import Packet, BleStream


In [58]:
pkt = Packet()
pkt.from_string("D6BE898E420E146A6291F5C907FF5C001202F001029DCD", parse_mode='tolerant')
pkt.update()
pkt.to_string()

'D6BE898E420E146A6291F5C907FF5C001202F001B1020D'

In [5]:
import pickle
import pandas as pd


MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)

In [6]:
from modeling.BleDataset import BLEStreamDataset

pkt_df_val = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_test.parquet")
seq_table_val = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_test.parquet")
stream_idx_val = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_test.parquet")




dataset_val = BLEStreamDataset(packet_df=pkt_df_val, sequence_table=seq_table_val, stream_index=stream_idx_val,
                           max_sequence_length=32, max_token_length=1024,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='validation', augment_data=True, adapt_sequence_length=True, deterministic=False)

In [19]:
pkt_df_val[pkt_df_val['Label'] == 'Tile Tracker (lost)']

,Time,Time Delta,Source,Channel,RSSI,Hex Data,Label,File,Sequence_ID,Packet_in_Sequence
1135554,1.713228e+09,0,DB7E1B05AF8E,38,37,D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F6...,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),4007,0
1135555,1.713228e+09,725,DB7E1B05AF8E,39,42,D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F6...,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),4007,1
1135556,1.713228e+09,3904255,DB7E1B05AF8E,37,36,D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F6...,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),4007,2
1135557,1.713228e+09,724,DB7E1B05AF8E,38,37,D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F6...,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),4007,3
1135558,1.713228e+09,725,DB7E1B05AF8E,39,42,D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F6...,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),4007,4
...,...,...,...,...,...,...,...,...,...,...
1154902,1.771674e+09,792,DC90654758FA,38,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6344
1154903,1.771674e+09,793,DC90654758FA,39,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6345
1154904,1.771674e+09,1999212,DC90654758FA,37,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6346
1154905,1.771674e+09,793,DC90654758FA,38,32,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6347


In [7]:
data = pd.read_parquet((dataPathProcessed + "Inference_V3.parquet"))

In [8]:
data.sort_values(by='Time', ascending=True, inplace=True)
data['Time Delta'] = data.groupby("Source")['Time'].diff()
data['Time Delta'] = data['Time Delta'].fillna(0)
data['Time Delta'] = (data['Time Delta'] * 1_000_000).astype(int)
data['Time Delta'] = data['Time Delta'].apply(lambda x: min(x, 2**32-1))
data = data[data['Source'] != '']

In [9]:
data.groupby("Source").size()

Source
0006BA5C8951    1
0008ED582A0B    1
000C41FDDC26    1
0017B615D821    1
001A956F016C    1
               ..
FFF858AF35FE    3
FFF90EE9249B    1
FFFA85479F0E    1
FFFBBB0EF8DF    1
FFFBCC3FF20D    1
Length: 57592, dtype: int64

In [10]:
dataset_val[423]

{'tokens': tensor([  4,  28, 117,  ...,   0,   0,   0]),
 'target': tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 'label': 'Apple Find My Device (online)',
 'label_id': 2,
 'masked packets': [(1601983,
   38,
   'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (6588912, 39, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (1915837, 37, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (2016982, 39, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (5726057, 38, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (1526333, 39, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (11862826, 37, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (1092406, 38, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (1024210, 39, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (9034560, 37, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (9274532, 38, 'D6BE898E420E5FF443B9044507FF4C001202D25BA55783'),
  (2146031, 39, 'D6BE898E420E5FF443B90445

In [11]:
import torch
from modeling.HydraBLE import HydraBLETransformer
from bpe.bpe import BleBytePairEncoder

bpe = BleBytePairEncoder.from_state_dict(state_dict)

model = HydraBLETransformer(
        vocab_size=bpe.vocab_size,
        num_classes=15,
        pad_id=1,
        max_heads=8,
        head_dim=64,
        depth=4,
        max_len=2048,
        subnet_heads=(1, 2, 4, 8),
        separate_cls_heads=True,
    )


ckpt = torch.load(str(Path.cwd()) + r"\out\modeling\checkpoints\classification_finetuning\\"  + "best.pt", map_location="cuda:1")
model.load_state_dict(ckpt["model_state_dict"])
model.set_active_heads(8)
model.eval()
model.to('cuda:1')

HydraBLETransformer(
  (token_embed): Embedding(2048, 512)
  (blocks): ModuleList(
    (0-3): 4 x HydraEncoderBlock(
      (norm1): HydraLayerNorm()
      (attn): HydraAttention(
        (qkv): HydraLinear()
        (proj): HydraLinear()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): HydraLayerNorm()
      (mlp): HydraMlp(
        (fc1): HydraLinear()
        (fc2): HydraLinear()
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): HydraLayerNorm()
  (lm_head): HydraLinear()
  (cls_heads): ModuleDict(
    (1): HydraLinear()
    (2): HydraLinear()
    (4): HydraLinear()
    (8): HydraLinear()
  )
)

In [12]:
from data_processing.LabelLut import LABEL_OTHER_DEVICE


pkt_df_test = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_test.parquet")

known_labels = sorted(list(pkt_df_test['Label'].unique()))
known_labels.remove(LABEL_OTHER_DEVICE)

label_lut = {i: label for i, label in enumerate(known_labels)}
label_id_unknown = len(label_lut)
label_lut[label_id_unknown] = LABEL_OTHER_DEVICE

In [13]:
data[data['Source'] == "C9F06B6B6A11"]

,Time,Source,Channel,RSSI,Hex Data,Time Delta
6,1.776168e+09,C9F06B6B6A11,39,80,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,0
2118,1.776168e+09,C9F06B6B6A11,39,80,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,1910788
9841,1.776168e+09,C9F06B6B6A11,38,75,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,5554241
9842,1.776168e+09,C9F06B6B6A11,39,82,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,0
20521,1.776168e+09,C9F06B6B6A11,38,81,D6BE898E420E116A6B6BF0C907FF83E00A45CA0D486CB7,12064345
28022,1.776168e+09,C9F06B6B6A11,39,70,D6BE898E420E116A6B6BF0C907FF4C00120600015843F7,10917842
40259,1.776168e+09,C9F06B6B6A11,37,75,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,15070012
40260,1.776168e+09,C9F06B6B6A11,38,74,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,0
40261,1.776168e+09,C9F06B6B6A11,39,77,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,1009
57141,1.776168e+09,C9F06B6B6A11,37,75,D6BE898E420E116A6B6BF0C907FF4C00120200015843F7,13187954


In [14]:
import random


def random_hex(length):
    hex_chars = "0123456789abcdef"
    return ''.join(random.choice(hex_chars) for _ in range(length)).upper()

In [66]:
import itertools
from tqdm import tqdm


result = []

for source, g in tqdm(data.groupby("Source")):

    hex_data = list(g['Hex Data'])[:200]
    channel = list(g['Channel'])[:200]
    time_delta = list(g['Time Delta'])[:200]

    pkts = [(t, c, p) for (t, c, p) in zip(time_delta, channel, hex_data)]


    #pkt_string = "D6BE898E420E116A6B6BF0C907FF4C00120200015843F7"
    pkt_string = 'D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F64F578C64D2BC0303EDFE108420'
    #pkt_string = "D6BE898E420EEC867E79ED5F07FF4C001202A679A55783"
    #print(len(find_my_online))
    start = 20
    end = 40
    #pkt_string = pkt_string[:start] + find_my_online[start:end] + pkt_string[end:]

    pkts = [(0, 37, pkt_string)]



    tokens = bpe.encode_many(pkts)
    tokens = list(itertools.chain(*tokens))
    tokens = tokens[:1024]
    length = len(tokens)
    print(tokens[:20])
    tokens = tokens + [bpe.PAD_ID for _ in range(1024 -length)]

    assert len(tokens) == 1024

    tokens = torch.tensor(tokens).unsqueeze(0).to('cuda:1')

    with torch.no_grad():
        logits = model(tokens)
        logits = logits.cpu()

        probas = torch.nn.functional.softmax(logits, dim=-1)
        probas = probas.squeeze(0)

        max_proba, _ = torch.max(probas, dim=-1)

        if float(max_proba) < 0.3:
            result.append([source, LABEL_OTHER_DEVICE, float(max_proba)])
            print([source, LABEL_OTHER_DEVICE, float(max_proba)])

        else:
            pred = torch.argmax(probas, dim=-1)
            proba_cls = probas[int(pred)]
            pred = label_lut[int(pred)]
            print([source, pred, float(proba_cls)])

            if proba_cls > 0.95:
                print([source, pred, float(proba_cls)])
                print(g['Hex Data'].iloc[0])
            result.append([source, pred, float(proba_cls)])
    break


  0%|          | 0/57592 [00:00<?, ?it/s]

[4, 4, 4, 4, 516, 561, 402, 435, 265, 287, 386, 479, 605, 359, 506, 339, 347, 400, 360, 470]
['0006BA5C8951', 'Tile Tracker (lost)', 0.9968600273132324]
['0006BA5C8951', 'Tile Tracker (lost)', 0.9968600273132324]
D6BE898E600951895CBA06000231109AFEC7


  0%|          | 0/57592 [00:00<?, ?it/s]


In [65]:
from ble import Packet, BleStream
apple_inference = "D6BE898E420E146A6291F5C907FF4C00120200018607CD"
apple_inference_masked = "D6BE898E420E146A6291F5C907FF4C001202F001029DCD"
apple_inference_masked_vendor  = "D6BE898E420E146A6291F5C907FF5C001202F001B1020D"

tile = "D6BE898E601BD0334998150714FF7C0112345678901234567890123456789012346737D3"
tile_real = "D6BE898E601B8EAF051B7EDB0201060D160DFE020063F64F578C64D2BC03030DFEE25D40"
tile_other = 'D6BE898E601C8EAF051B7EDB0201060E160DFEAB020063F64F578C64D2BC03030DFEA15040'
tile_genuine = 'D6BE898E601B8EAF051B7EDB0201060D16EDFE020063F64F578C64D2BC0303EDFE108420'

pkts = [apple_inference, apple_inference_masked, apple_inference_masked_vendor, tile, tile_real, tile_other, tile_genuine]
stream = BleStream()
for p in pkts:
    pkt = Packet()
    print(pkt)
    pkt.from_string(p, parse_mode='normal')
    stream.add_packet(pkt)
stream.to_pcap_file("interesting_packets.pcapng")

7it [00:00, 3500.25it/s]
